# Topic 17 — Pipelines
### Theory → the leakage problem, again → sklearn Pipeline → ColumnTransformer → experiment.

You've now seen scaling (15), imputation/encoding (16), and been reminded repeatedly to
"fit on train only, transform test" to avoid **data leakage** (Topic 5). A `Pipeline` bundles every
preprocessing step + the final model into ONE object, so that rule is enforced automatically —
you literally can't leak by accident once your steps are inside a pipeline.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score

rng = np.random.default_rng(0)

## 1. The manual (leaky-if-you're-not-careful) way

This is what you'd do without pipelines — it WORKS if done carefully, but it's easy to
accidentally call `.fit_transform()` on the full dataset before splitting, which silently leaks
test-set information into training.

In [ ]:
X = rng.normal(50, 15, size=(200, 3))
y = (X[:, 0] + X[:, 1] * 0.5 > 50).astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Correct manual approach: fit scaler on TRAIN only
scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)      # transform, not fit_transform!

model = LogisticRegression().fit(X_train_scaled, y_train)
print("manual approach accuracy:", accuracy_score(y_test, model.predict(X_test_scaled)))
# This is correct, but every new preprocessing step means one more place you could forget "fit on train only".

## 2. The same thing, with `Pipeline`

A `Pipeline` chains steps together. Calling `.fit()` on the pipeline fits EVERY step only on
whatever data you pass in — so `pipeline.fit(X_train, y_train)` can never see X_test, by construction.

In [ ]:
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression()),
])

pipe.fit(X_train, y_train)          # scaler fits on X_train only, internally
print("pipeline accuracy:", accuracy_score(y_test, pipe.predict(X_test)))
# Same result as the manual version -- but now it's structurally impossible to leak.

## 3. Why this matters even more with cross-validation

Recall Topic 6: cross-validation retrains on many different train/val splits. Doing manual scaling
correctly across every fold is tedious and error-prone. A pipeline makes it trivial and safe.

In [ ]:
scores = cross_val_score(pipe, X, y, cv=5)
print("cross-val scores:", scores)
print("mean accuracy:", scores.mean())
# cross_val_score automatically re-fits the WHOLE pipeline (including the scaler) fresh on each
# fold's training portion -- no leakage across any of the 5 folds, with zero extra code from you.

## 4. `ColumnTransformer` — different preprocessing for different columns

Real datasets mix numeric and categorical columns, which need different preprocessing
(scaling for numeric, one-hot encoding for categorical). `ColumnTransformer` applies the right
transformer to the right columns, all inside one pipeline.

In [ ]:
df = pd.DataFrame({
    "age": [25, 30, np.nan, 45, 22, 38],
    "platform": ["insta", "twitter", "insta", "youtube", "twitter", "insta"],
    "text_length": [50, 120, 30, 200, 45, 90],
})
y_df = np.array([1, 0, 1, 0, 1, 0])

numeric_features = ["age", "text_length"]
categorical_features = ["platform"]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline([
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

full_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", LogisticRegression()),
])

full_pipeline.fit(df, y_df)
print("predictions on training data:", full_pipeline.predict(df))
print("pipeline structure:")
print(full_pipeline)

## 5. Reproducibility benefit

A fitted pipeline is a single object you can save (Topic 42) and reload — it carries its exact
fitted scalers/encoders/imputer values with it, guaranteeing identical preprocessing every time
you run inference, without needing to remember or re-derive any statistics by hand.

In [ ]:
import joblib

joblib.dump(full_pipeline, "pipeline.joblib")
loaded_pipeline = joblib.load("pipeline.joblib")

print("loaded pipeline predictions match:", np.array_equal(
    full_pipeline.predict(df), loaded_pipeline.predict(df)
))

## Exercise

In [ ]:
# --- Try it yourself ---
# 1. Add a SimpleImputer(strategy="constant", fill_value="unknown") step for the categorical_features
#    to handle missing platform values too, and rebuild the ColumnTransformer.
# 2. Swap LogisticRegression for a RandomForestClassifier (Topic 13) inside full_pipeline --
#    everything else should keep working unchanged.
# 3. Run cross_val_score on full_pipeline using df and y_df with cv=3 -- note any errors from the
#    tiny dataset size and think about why (hint: very few samples per class per fold).
# 4. In one sentence: why does putting preprocessing INSIDE the pipeline matter more once you
#    start doing hyperparameter search (GridSearchCV, Topic 39) across many train/val splits?

---
### Next up: **Topic 18 — Unsupervised Learning: Clustering** (K-Means, hierarchical, DBSCAN).

Say "next" when you're ready.